# Project 1:  Customer Sales Analytics using Snowflake Cloud Data Warehouse
- ## Problem Statement:
  An online retail company has recently migrated its operational database to the Snowflake Cloud Data Warehouse     to improve reporting, business intelligence, and decision-making.

In [ ]:
%%sql -r dataframe_1
-- Create Warehouse
CREATE OR REPLACE WAREHOUSE SALES_WH
WITH
WAREHOUSE_SIZE = 'XSMALL'
AUTO_SUSPEND = 60
AUTO_RESUME = TRUE;

In [ ]:
%%sql -r dataframe_2
USE WAREHOUSE SALES_WH;

In [ ]:
%%sql -r dataframe_3
-- Create Database
CREATE OR REPLACE DATABASE CUSTOMER_SALES_DB;

In [ ]:
%%sql -r dataframe_4
-- Create Schema
CREATE OR REPLACE SCHEMA CUSTOMER_SALES_DB.SALES_SCHEMA;

In [ ]:
%%sql -r dataframe_5
USE DATABASE CUSTOMER_SALES_DB;
USE SCHEMA SALES_SCHEMA;


In [ ]:
%%sql -r dataframe_6
CREATE OR REPLACE FILE FORMAT CSV_FORMAT
TYPE = CSV
FIELD_DELIMITER = ','
SKIP_HEADER = 1;

In [ ]:
%%sql -r dataframe_7
-- Create Internal Stage
CREATE OR REPLACE STAGE SALES_STAGE
FILE_FORMAT = CSV_FORMAT;

In [ ]:
%%sql -r dataframe_8
SHOW WAREHOUSES;
SHOW DATABASES;
SHOW SCHEMAS;
SHOW FILE FORMATS;
SHOW STAGES;
LIST @SALES_STAGE;

In [ ]:
%%sql -r dataframe_9
-- Create CUSTOMERS table
CREATE OR REPLACE TABLE CUSTOMERS (
    CUSTOMER_ID INT,
    FIRST_NAME STRING,
    LAST_NAME STRING,
    EMAIL STRING,
    PHONE STRING,
    ADDRESS STRING
);

In [ ]:
%%sql -r dataframe_10
-- Create FOODITEMS table
CREATE OR REPLACE TABLE FOODITEMS (
    FOOD_ID INT,
    NAME STRING,
    PRICE NUMBER(10,2),
    CATEGORY STRING,
    AVAILABILITY STRING
);

In [ ]:
%%sql -r dataframe_11
-- Create ORDERS table
CREATE OR REPLACE TABLE ORDERS (
    ORDER_ID INT,
    CUSTOMER_ID INT,
    FOOD_ID INT,
    QUANTITY INT,
    ORDER_DATE TIMESTAMP,
    STATUS STRING,
    TOTAL_AMOUNT NUMBER(10,2)
);


In [ ]:
%%sql -r dataframe_12
-- Load data
COPY INTO CUSTOMERS
FROM @SALES_STAGE/customers.csv
FILE_FORMAT = CSV_FORMAT;

COPY INTO FOODITEMS
FROM @SALES_STAGE/fooditems.csv
FILE_FORMAT = CSV_FORMAT;

COPY INTO ORDERS
FROM @SALES_STAGE/orders.csv
FILE_FORMAT = CSV_FORMAT;

In [ ]:
%%sql -r dataframe_13
SELECT * FROM CUSTOMERS;
SELECT * FROM FOODITEMS;
SELECT * FROM ORDERS;

In [ ]:
%%sql -r dataframe_14
SELECT
    C.CUSTOMER_ID,
    CONCAT(C.FIRST_NAME,' ',C.LAST_NAME) AS CUSTOMER_NAME,
    SUM(O.TOTAL_AMOUNT) AS TOTAL_SPENT
FROM CUSTOMERS C
JOIN ORDERS O
ON C.CUSTOMER_ID=O.CUSTOMER_ID
GROUP BY C.CUSTOMER_ID,C.FIRST_NAME,C.LAST_NAME
ORDER BY TOTAL_SPENT DESC;

In [ ]:
%%sql -r dataframe_15
SELECT
    C.CUSTOMER_ID,
    CONCAT(C.FIRST_NAME,' ',C.LAST_NAME) AS CUSTOMER_NAME,
    SUM(O.TOTAL_AMOUNT) AS TOTAL_SPENT
FROM CUSTOMERS C
JOIN ORDERS O
ON C.CUSTOMER_ID=O.CUSTOMER_ID
GROUP BY C.CUSTOMER_ID,C.FIRST_NAME,C.LAST_NAME
ORDER BY TOTAL_SPENT DESC
LIMIT 1;

In [ ]:
%%sql -r dataframe_16
SELECT SUM(TOTAL_AMOUNT) AS TOTAL_REVENUE
FROM ORDERS;

In [ ]:
%%sql -r dataframe_17
SELECT
    F.CATEGORY,
    SUM(O.TOTAL_AMOUNT) AS REVENUE
FROM FOODITEMS F
JOIN ORDERS O
ON F.FOOD_ID=O.FOOD_ID
GROUP BY F.CATEGORY
ORDER BY REVENUE DESC;

In [ ]:
%%sql -r dataframe_18
SELECT STATUS,
       SUM(TOTAL_AMOUNT) AS REVENUE
FROM ORDERS
GROUP BY STATUS;

In [ ]:
%%sql -r dataframe_19
SELECT
    CONCAT(C.FIRST_NAME,' ',C.LAST_NAME) AS CUSTOMER_NAME,
    SUM(O.TOTAL_AMOUNT) AS TOTAL_SPENT
FROM CUSTOMERS C
JOIN ORDERS O
ON C.CUSTOMER_ID=O.CUSTOMER_ID
GROUP BY C.FIRST_NAME,C.LAST_NAME
ORDER BY TOTAL_SPENT DESC
LIMIT 3;

In [ ]:
%%sql -r dataframe_20
SELECT
    C.CUSTOMER_ID,
    CONCAT(C.FIRST_NAME,' ',C.LAST_NAME) AS CUSTOMER_NAME,
    COUNT(O.ORDER_ID) AS ORDERS_PLACED
FROM CUSTOMERS C
JOIN ORDERS O
ON C.CUSTOMER_ID=O.CUSTOMER_ID
GROUP BY C.CUSTOMER_ID,C.FIRST_NAME,C.LAST_NAME
ORDER BY ORDERS_PLACED DESC;

In [ ]:
%%sql -r dataframe_21
SELECT
    ORDER_ID,
    CUSTOMER_ID,
    FOOD_ID,
    STATUS,
    TOTAL_AMOUNT
FROM ORDERS
WHERE STATUS='Delivered';

In [ ]:
%%sql -r dataframe_22
SELECT
    O.ORDER_ID,
    CONCAT(C.FIRST_NAME,' ',C.LAST_NAME) AS CUSTOMER_NAME,
    O.ORDER_DATE,
    O.STATUS,
    O.TOTAL_AMOUNT
FROM ORDERS O
JOIN CUSTOMERS C
ON O.CUSTOMER_ID=C.CUSTOMER_ID
WHERE O.ORDER_DATE > '2026-07-12'
ORDER BY O.ORDER_DATE;

In [ ]:
%%sql -r dataframe_23
CREATE OR REPLACE VIEW CUSTOMER_SALES_REPORT AS
SELECT
    C.CUSTOMER_ID,
    CONCAT(C.FIRST_NAME, ' ', C.LAST_NAME) AS CUSTOMER_NAME,
    SUM(O.TOTAL_AMOUNT) AS TOTAL_SPENT
FROM CUSTOMERS C
INNER JOIN ORDERS O
    ON C.CUSTOMER_ID = O.CUSTOMER_ID
GROUP BY
    C.CUSTOMER_ID,
    C.FIRST_NAME,
    C.LAST_NAME;


In [ ]:
%%sql -r dataframe_26
SHOW VIEWS;

In [ ]:
%%sql -r dataframe_24
SELECT * FROM CUSTOMER_SALES_REPORT;

In [ ]:
%%sql -r dataframe_25
SELECT *
FROM CUSTOMER_SALES_REPORT
ORDER BY TOTAL_SPENT DESC;